# yarn-niah wave1b long7B far past its window: 3 depths only, because prefill at 128k dominates and a kernel that times out returns nothing.Preset `wave1b` on `NvidiaTeslaT4`, harness pinned to `90b4616af24363c57471611b5ac0e10967825a45`.| setting | value ||---|---|| model | `Qwen/Qwen2.5-7B-Instruct` || lengths | `65536,131072` || depths | `0.1,0.5,0.9` || extra flags | `--load-in-4bit --attn sdpa` || max new tokens | 64 |Both arms run the same grid, so the only variable is `rope_scaling`.

In [ ]:
# 1) Harness at a pinned commit. Pinning matters: these numbers get compared#    against a later run, and "main moved" is not an explanation anyone wants.import os, subprocess, sysREPO_URL = "https://github.com/Nikita-prog-art/yarn-haystack.git"COMMIT   = "90b4616af24363c57471611b5ac0e10967825a45"WORK     = "/kaggle/working"os.makedirs(WORK, exist_ok=True)os.chdir(WORK)if not os.path.isdir("yarn-haystack"):    subprocess.run(["git", "clone", "-q", REPO_URL, "yarn-haystack"], check=True)os.chdir("yarn-haystack")subprocess.run(["git", "fetch", "-q", "--all"], check=True)subprocess.run(["git", "checkout", "-q", COMMIT], check=True)head = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()assert head == COMMIT, f"harness is at {head}, expected {COMMIT}"sys.path.insert(0, os.getcwd())print("harness commit:", head)

In [ ]:
# 2) Install only what is missing. Kaggle ships a CUDA-matched torch, and#    upgrading matplotlib/datasets provokes conflicts with preinstalled tools.import importlib.metadata as md, subprocess, sysdef version(p):    try:        return md.version(p)    except md.PackageNotFoundError:        return Nonedef too_old(v, floor):    return v is None or tuple(int(x) for x in v.split(".")[:2]) < floorto_install = []if too_old(version("transformers"), (4, 51)):    to_install.append("transformers>=4.51")for p in ("accelerate", "datasets", "bitsandbytes"):    if version(p) is None:        to_install.append(p)if to_install:    print("pip install:", " ".join(to_install), flush=True)    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", *to_install], check=True)for p in ("transformers", "torch", "accelerate", "datasets", "bitsandbytes", "matplotlib"):    print(f"  {p:14s} {version(p)}")import torchprint(f"\ntorch {torch.__version__} | cuda {torch.cuda.is_available()} | gpus {torch.cuda.device_count()}")for i in range(torch.cuda.device_count()):    pr = torch.cuda.get_device_properties(i)    print(f"  gpu{i}: {pr.name}, {pr.total_memory / 2**30:.1f} GiB")assert torch.cuda.is_available(), "no CUDA: this kernel was pushed without a GPU"_v = version("transformers")assert _v and tuple(int(x) for x in _v.split(".")[:2]) >= (4, 51), f"need transformers>=4.51, have {_v}"

In [ ]:
# 3) Preflight: internet (the usual cause of a one-second death) and the corpus.import os, urllib.requestimport niahtry:    print("huggingface.co ->", urllib.request.urlopen("https://huggingface.co", timeout=20).status)except Exception as exc:    raise SystemExit(f"no internet in this kernel: {exc!r} (Kaggle needs phone verification for that)")text = niah.load_corpus(None, os.path.join(os.getcwd(), "cache"))print(f"corpus OK: {len(text)/1e6:.1f}M chars cached")

In [ ]:
# 4) Run both arms under the "wave1b" preset, streaming output live.import collections, subprocess, sysdef niah(*args):    """-u matters: through a pipe the child's stdout is block-buffered and a long    128k case otherwise looks frozen. The tail goes into the exception, because a    bare CalledProcessError hides the one thing worth reading."""    cmd = [sys.executable, "-u", "run.py", *args]    print("$", " ".join(cmd), flush=True)    tail = collections.deque(maxlen=40)    with subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,                          text=True, bufsize=1) as p:        for line in p.stdout:            print(line, end="", flush=True)            tail.append(line.rstrip())        p.wait()    if p.returncode:        print(f"!! exit {p.returncode} - keeping results collected so far")COMMON = ["--model", "Qwen/Qwen2.5-7B-Instruct", "--lengths", "65536,131072", "--depths", "0.1,0.5,0.9",          "--max-new-tokens", "64", *['--load-in-4bit', '--attn', 'sdpa']]for arm, extra in (("base", []), ("yarn", ["--factor", "4", "--original-max", "32768"])):    niah(*COMMON, "--arm", arm, *extra, "--out", f"results/{arm}_wave1b.jsonl")

In [ ]:
# 5) Report. analyze.py is robust to a partial run, so this is worth doing even if#    an arm was cut short: you get the heatmap for whatever finished.import glob, subprocess, sys, osfrom IPython.display import Image, Markdown, displayres = sorted(glob.glob("results/*.jsonl"))print("result files:", res)if not res:    raise SystemExit("no results were produced - read the output above")subprocess.run([sys.executable, "analyze.py", *res, "--outdir", "report"], check=True)display(Markdown(open("report/summary.md", encoding="utf-8").read()))if os.path.exists("report/heatmaps.png"):    display(Image("report/heatmaps.png"))print("\nfiles under /kaggle/working (fetched by `kaggle kernels output`):")for root, _dirs, files in os.walk("results"):    for f in sorted(files):        print("  ", os.path.join(root, f))